# Portfolio Evaluation
Compares the SWDA+XMME blended portfolio against two benchmarks:
- **Benchmark 1**: VWCE (Vanguard FTSE All-World Acc)
- **Benchmark 2**: FTSE All-World index (proxied by VWRL, same index, distributing share class, longer history)

In [ ]:
%pip install -q yfinance matplotlib

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from datetime import datetime

In [ ]:
# --- Configuration ---
TICKERS = {
    'SWDA': 'SWDA.L',
    'XMME': 'XMME.DE',
    'VWCE': 'VWCE.DE',
    'FTSE_AW': 'VWRL.L',   # FTSE All-World proxy (distributing, same index as VWCE)
}

LOG_SHEET_NAME = 'portfolio_weights_log'

# If True, months before the first log entry use the first recorded weights.
# If False, analysis starts only from the first log entry.
BACKFILL_WITH_FIRST_WEIGHTS = True

In [ ]:
# --- Load weights log from Google Sheets ---
def get_gspread_client():
    scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
    credentials_json = json.loads(os.environ['GOOGLE_CREDENTIALS'])
    credentials = ServiceAccountCredentials.from_json_keyfile_dict(credentials_json, scope)
    return gspread.authorize(credentials)

client = get_gspread_client()
sheet = client.open(LOG_SHEET_NAME).sheet1
records = sheet.get_all_records()

weights_log = pd.DataFrame(records)
weights_log['Date'] = pd.to_datetime(weights_log['Date'])
weights_log = weights_log.set_index('Date').sort_index()

print(weights_log)

In [ ]:
# --- Download monthly price data ---
all_tickers = list(TICKERS.values())
raw = yf.download(all_tickers, period='max', interval='1mo', auto_adjust=True, progress=False)
prices = raw['Close'].copy()
prices.columns = {v: k for k, v in TICKERS.items()}  # rename back to short names
prices = prices.rename(columns={v: k for k, v in TICKERS.items()})
prices.index = prices.index.tz_localize(None).to_period('M').to_timestamp('M')
prices = prices.dropna(how='all')
print(f'Price data: {prices.index[0].date()} → {prices.index[-1].date()}, {len(prices)} months')
prices.tail()

In [ ]:
# --- Assign monthly weights to each period ---
# For each month in price history, use the most recent weight logged up to that month.
monthly_returns = prices[['SWDA', 'XMME']].pct_change().dropna()

def get_active_weight(date, log):
    past = log[log.index <= date]
    if past.empty:
        return None
    return past.iloc[-1][['SWDA', 'XMME']].values

if BACKFILL_WITH_FIRST_WEIGHTS:
    first_weights = weights_log.iloc[0][['SWDA', 'XMME']].values
    active_weights = {
        date: (get_active_weight(date, weights_log) if get_active_weight(date, weights_log) is not None else first_weights)
        for date in monthly_returns.index
    }
else:
    first_log_date = weights_log.index[0]
    monthly_returns = monthly_returns[monthly_returns.index >= first_log_date]
    active_weights = {date: get_active_weight(date, weights_log) for date in monthly_returns.index}

portfolio_returns = pd.Series(
    {date: float(np.dot(active_weights[date], monthly_returns.loc[date].values))
     for date in monthly_returns.index},
    name='Portfolio'
)

print('Portfolio monthly returns computed:', len(portfolio_returns), 'months')

In [ ]:
# --- Align benchmarks and compute cumulative returns ---
bench_returns = prices[['VWCE', 'FTSE_AW']].pct_change().reindex(portfolio_returns.index).dropna()
port_aligned = portfolio_returns.reindex(bench_returns.index)

all_returns = pd.concat([port_aligned, bench_returns], axis=1).dropna()
all_returns.columns = ['Portfolio (SWDA+XMME)', 'VWCE', 'FTSE All-World (VWRL proxy)']

cumulative = (1 + all_returns).cumprod()
cumulative = cumulative / cumulative.iloc[0]  # rebase to 1

cumulative.tail()

In [ ]:
# --- Plot cumulative returns ---
fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for col, color in zip(cumulative.columns, colors):
    ax.plot(cumulative.index, cumulative[col], label=col, color=color, linewidth=1.8)

ax.set_title('Cumulative Return: Portfolio vs Benchmarks')
ax.set_ylabel('Growth of 1 unit invested')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# --- Performance metrics ---
def max_drawdown(cum_series):
    roll_max = cum_series.cummax()
    drawdown = (cum_series - roll_max) / roll_max
    return drawdown.min()

n_months = len(all_returns)
n_years = n_months / 12

metrics = {}
for col in all_returns.columns:
    r = all_returns[col]
    cum = cumulative[col]
    total_return = cum.iloc[-1] - 1
    cagr = (1 + total_return) ** (1 / n_years) - 1
    vol = r.std() * np.sqrt(12)
    sharpe = cagr / vol if vol > 0 else np.nan
    mdd = max_drawdown(cum)
    metrics[col] = {
        'Total Return': f'{total_return:.1%}',
        'CAGR': f'{cagr:.2%}',
        'Ann. Volatility': f'{vol:.2%}',
        'Sharpe (rf=0)': f'{sharpe:.2f}',
        'Max Drawdown': f'{mdd:.1%}',
    }

pd.DataFrame(metrics).T